# Final Data Preparation for Dashboarding

This notebook prepares the final aggregated tables and engineering features required for dashboard visualization. 

**Note on Added Value**: We have implemented a granular **Risk Score** (0 for Low, 1 for Medium, 2+ for High Risk) to allow for deeper segmented analysis.

In [1]:
import pandas as pd
import numpy as np

# ==============================
# 1. LOAD DATA
# ==============================
df = pd.read_csv('../data/processed/cleaned_loan_data.csv')

print(f"Initial shape: {df.shape}")

Initial shape: (8019, 21)


In [2]:
# Normalize Credit Scores > 850
df.loc[df['Credit Score'] > 850, 'Credit Score'] = df['Credit Score'] / 10

# Additive Risk Score Calculation
df['Risk Score'] = (
    (df['Number of Credit Problems'] > 0).astype(int) +
    (df['Bankruptcies'] > 0).astype(int) +
    (df['Tax Liens'] > 0).astype(int)
)

# Risk Category Logic based on Score
def risk_category(score):
    if score == 0:
        return 'Low Risk'
    elif score == 1:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['Risk Category'] = df['Risk Score'].apply(risk_category)

# Synthetic loan_status
df['loan_status'] = df['Risk Score'].apply(lambda x: 'Charged Off' if x > 0 else 'Fully Paid')

In [3]:
# Normalize Credit Scores > 850 (dividing by 10 to correct scaling issue)
df.loc[df['Credit Score'] > 850, 'Credit Score'] = df['Credit Score'] / 10

# NEW: Additive Risk Score Calculation
df['Risk Score'] = (
    (df['Number of Credit Problems'] > 0).astype(int) +
    (df['Bankruptcies'] > 0).astype(int) +
    (df['Tax Liens'] > 0).astype(int)
)

# Refined Risk Category Logic based on Score
def risk_label(score):
    if score == 0:
        return 'Low Risk'
    elif score == 1:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['Risk Category'] = df['Risk Score'].apply(risk_label)

# Synthetic loan_status (Keeping for Default Rate KPI compatibility)
# Logic: Charged Off if Risk Score > 0
df['loan_status'] = df['Risk Score'].apply(lambda x: 'Charged Off' if x > 0 else 'Fully Paid')

print("Refined Risk Category Distribution:")
print(df['Risk Category'].value_counts())
print(f"\nDefault Rate (Risk > 0): {(df['loan_status'] == 'Charged Off').mean():.2%}")

Refined Risk Category Distribution:
Risk Category
Low Risk       6941
High Risk       979
Medium Risk      99
Name: count, dtype: int64

Default Rate (Risk > 0): 13.44%


## 3. KPI Calculations

In [4]:
default_rate = (df['loan_status'] == 'Charged Off').mean()
avg_loan = df['Current Loan Amount'].mean()
avg_income = df['Annual Income'].mean()
avg_credit = df['Credit Score'].mean()

print("Default Rate:", default_rate)
print("Avg Loan:", avg_loan)
print("Avg Income:", avg_income)
print("Avg Credit Score:", avg_credit)

Default Rate: 0.13443072702331962
Avg Loan: 304735.34430727025
Avg Income: 1369106.0397805213
Avg Credit Score: 716.6312507793989


## 4. Feature Engineering (DTI)

In [5]:
# Debt-to-Income Ratio (DTI)
df['DTI'] = df['Monthly Debt'] / (df['Annual Income'] / 12)
print("DTI Stats:")
display(df['DTI'].describe())

DTI Stats:


count    8019.000000
mean        0.172618
std         0.080926
min         0.000000
25%         0.112001
50%         0.169999
75%         0.228999
max         0.399997
Name: DTI, dtype: float64

## 5. Segmented Analysis (Dashboard Tables)

In [6]:
# Default rate by Home Ownership
default_by_home = df.groupby('Home Ownership')['loan_status'].apply(
    lambda x: (x == 'Charged Off').mean()
).reset_index(name='default_rate')

# Loan amount by employment length
loan_by_emp = df.groupby('Years in current job')['Current Loan Amount'].mean().reset_index()

# Risk category aggregation
risk_summary = df.groupby('Risk Category').agg({
    'Current Loan Amount': 'mean',
    'Annual Income': 'mean',
    'DTI': 'mean',
    'Risk Score': 'count'
}).rename(columns={'Risk Score': 'Applicant Count'}).reset_index()

display(risk_summary)

,Risk Category,Current Loan Amount,Annual Income,DTI,Applicant Count
0,High Risk,266582.179775,1.297171e+06,0.163773,979
1,Low Risk,310578.656101,1.378986e+06,0.174137,6941
2,Medium Risk,272346.666667,1.387755e+06,0.153539,99


## 6. Export Dashboard Tables

In [7]:
final_dashboard = df.groupby(['Risk Category']).agg({
    'Current Loan Amount': 'mean',
    'Annual Income': 'mean',
    'Credit Score': 'mean',
    'DTI': 'mean'
}).reset_index()

# Save Files
final_dashboard.to_csv('../data/processed/final_dashboard_data.csv', index=False)
default_by_home.to_csv('../data/processed/default_by_home.csv', index=False)
loan_by_emp.to_csv('../data/processed/loan_by_emp.csv', index=False)
risk_summary.to_csv('../data/processed/risk_summary.csv', index=False)
df.to_csv('../data/processed/final_loan_risk_processed.csv', index=False)

print("All dashboard files with NEW Risk Scoring saved successfully!")

All dashboard files with NEW Risk Scoring saved successfully!


In [8]:
print(df.dtypes)

Loan ID                          object
Customer ID                      object
Current Loan Amount             float64
Term                             object
Credit Score                    float64
Annual Income                   float64
Years in current job             object
Home Ownership                   object
Purpose                          object
Monthly Debt                    float64
Years of Credit History         float64
Months since last delinquent    float64
Number of Open Accounts         float64
Number of Credit Problems       float64
Current Credit Balance          float64
Maximum Open Credit             float64
Bankruptcies                    float64
Tax Liens                       float64
DTI                             float64
Loan_to_Income                  float64
Risk Category                    object
Risk Score                        int64
loan_status                      object
dtype: object
